# Chapter 14 — Financial and Legal Domain Agents

**Book:** *30 Agents Every AI Engineer Must Build*
**Author:** Imran Ahmad
**Publisher:** Packt Publishing

---

## Overview

This notebook implements two production-grade agent architectures from Chapter 14:

1. **Financial Advisory Agent** (Section 14.1) — A supervised multi-agent system with market data analysis, risk assessment (VaR, CVaR, volatility scoring), personalized financial planning, and compliance-by-architecture validation gates.

2. **Legal Intelligence Agent** (Section 14.2) — A RAG-powered legal research system with hybrid retrieval, authority-weighted ranking, precedent finding, contract analysis, and citation verification.

### How This Notebook Works

- **Simulation Mode (default):** Press Enter at each API key prompt to run with high-fidelity mock data. No API keys required — every cell runs successfully.
- **Live Mode:** Provide your API keys (OpenAI, Finnhub, Tavily) for real API calls. The `@graceful_fallback` decorator ensures the notebook never crashes even if an API call fails.

### Technical Requirements (p.2)
- Python 3.10+
- Dependencies: See `requirements.txt` (install via `pip install -r requirements.txt`)


In [1]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 0: Setup & Configuration
# Chapter Ref: Technical Requirements (p.2)
# Book: 30 Agents Every AI Engineer Must Build — Imran Ahmad (Packt Publishing)
# ─────────────────────────────────────────────────────────────────────────────

import os
import sys
import json
import warnings
warnings.filterwarnings("ignore")

# Enable ANSI color codes on Windows (see troubleshooting.md T8)
if sys.platform == "win32":
    os.system("")

# ── Load environment variables ──
from dotenv import load_dotenv
load_dotenv()

# ── Import mocking & resilience layer ──
from mock_llm import (
    ColorLogger,
    ServiceConfig,
    graceful_fallback,
    MockChatOpenAI,
    MockStructuredChain,
    MockEmbeddingModel,
    MockVectorStore,
    logger,
)

# ── Import synthetic data ──
from mock_data import (
    MOCK_STOCK_DATA,
    MOCK_FINNHUB_QUOTES,
    MOCK_FINNHUB_FINANCIALS,
    generate_mock_price_history,
    MOCK_TAVILY_NEWS,
    MOCK_CLIENT_PROFILES,
    MOCK_LEGAL_CASES,
    MOCK_CONTRACT,
    MOCK_INTER_AGENT_MESSAGE,
)

# ── Detect API availability and print dashboard ──
config = ServiceConfig(interactive=True)

# ── Conditional LLM selection ──
# If OpenAI key is available, use the real ChatOpenAI; otherwise use MockChatOpenAI
if config.is_live("OPENAI_API_KEY"):
    from langchain_openai import ChatOpenAI
    llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)
    logger.success("Using LIVE OpenAI LLM (gpt-4o-mini)")
else:
    llm = MockChatOpenAI(model="mock-gpt-4o-mini", temperature=0)
    logger.warning("Using MockChatOpenAI (Simulation Mode)")

# ── Core imports for agent construction ──
import numpy as np
from typing import Any, Dict, List, Optional, TypedDict, Literal, Annotated
from pydantic import BaseModel, Field

from langchain_core.messages import AIMessage, HumanMessage, SystemMessage
from langchain_core.tools import tool

logger.success("All imports loaded successfully. Notebook is ready.")



════════════════════════════════════════════════════════
  CHAPTER 14 — SERVICE STATUS DASHBOARD
  Book: 30 Agents Every AI Engineer Must Build
  Author: Imran Ahmad
════════════════════════════════════════════════════════
  OpenAI (LLM)                    ○ SIMULATED
  Finnhub (Financial Data)        ○ SIMULATED
  Tavily (News Search)            ○ SIMULATED
════════════════════════════════════════════════════════

[13:22:13] [Chapter14] WARNING Using MockChatOpenAI (Simulation Mode)


[13:22:14] [Chapter14] SUCCESS All imports loaded successfully. Notebook is ready.


---

## Section 14.1 — Financial Advisory Agent

### Supervisor Architecture (Fig 14.1)

The Financial Advisory Agent uses a **supervisor pattern** where a central supervisor agent orchestrates specialized sub-agents:

```
                    ┌─────────────────┐
                    │   Supervisor    │
                    │   Agent         │
                    └───────┬─────────┘
                            │
              ┌─────────────┼─────────────┐
              ▼             ▼             ▼
        ┌───────────┐ ┌───────────┐ ┌───────────┐
        │  Market   │ │ Analysis  │ │   News    │
        │  Data     │ │  Agent    │ │  Agent    │
        │  Agent    │ │           │ │           │
        └───────────┘ └───────────┘ └───────────┘
```

The supervisor decides which agent to invoke next, collects results, and determines when the workflow is complete. This prevents agents from acting independently and ensures coordinated execution.

> **Note (Knight Capital Incident):** Section 14.1 highlights the Knight Capital incident of 2012, where an automated trading system without proper oversight caused $440 million in losses in 45 minutes. The supervisor pattern with compliance gates is a direct architectural response to this type of risk.


In [2]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 1: Supervisor Architecture
# Chapter Ref: Section 14.1, Fig 14.1
# Book: 30 Agents Every AI Engineer Must Build — Imran Ahmad (Packt Publishing)
# ─────────────────────────────────────────────────────────────────────────────

logger.info("Building Supervisor Architecture (Sec 14.1, Fig 14.1)")

# ── Define the routing response schema ──
# The supervisor uses this to decide which agent to call next

class RouteResponse(BaseModel):
    """Supervisor routing decision.

    The supervisor returns this schema to indicate which agent should
    handle the next step, or 'FINISH' to end the workflow.

    Chapter Ref: Section 14.1, Fig 14.1
    Author: Imran Ahmad
    """
    next: Literal[
        "market_data_agent",
        "analysis_agent",
        "news_agent",
        "FINISH"
    ] = Field(
        description="The next agent to route to, or FINISH to end the workflow."
    )

# ── Supervisor system prompt ──
SUPERVISOR_SYSTEM_PROMPT = """You are the Financial Advisory Supervisor Agent.
Your role is to orchestrate a team of specialized agents to answer financial queries.

Available agents:
- market_data_agent: Fetches real-time market data (prices, fundamentals, volumes)
- analysis_agent: Performs portfolio analysis and risk assessment
- news_agent: Retrieves and summarizes relevant financial news

Workflow:
1. First, route to market_data_agent to gather current market data
2. Then, route to analysis_agent for portfolio and risk analysis
3. Then, route to news_agent for relevant news context
4. Finally, return FINISH when all information has been gathered

Always follow this sequence to ensure comprehensive analysis.
Book: 30 Agents Every AI Engineer Must Build — Imran Ahmad
Chapter: 14, Section 14.1
"""

# ── Build the supervisor chain ──
@graceful_fallback(
    fallback_value=RouteResponse(next="FINISH"),
    section_ref="Sec 14.1"
)
def supervisor_agent(state: dict) -> RouteResponse:
    """Supervisor agent that routes to the next specialized agent.

    Uses structured output to return a RouteResponse indicating which
    agent to invoke next in the workflow.

    Chapter Ref: Section 14.1, Fig 14.1
    Author: Imran Ahmad
    """
    messages = state.get("messages", [])
    supervisor_chain = llm.with_structured_output(RouteResponse)
    response = supervisor_chain.invoke(
        [SystemMessage(content=SUPERVISOR_SYSTEM_PROMPT)] + messages
    )
    logger.info(f"Supervisor routing decision: {response.next}")
    return response

# ── Test the supervisor ──
test_state = {"messages": [HumanMessage(content="Analyze AAPL and MSFT for a moderate growth portfolio")]}
route = supervisor_agent(test_state)
logger.success(f"Supervisor test: routed to '{route.next}'")


[13:22:14] [Chapter14] INFO Building Supervisor Architecture (Sec 14.1, Fig 14.1)
[13:22:14] [Chapter14] INFO Supervisor routing decision: market_data_agent
[13:22:14] [Chapter14] SUCCESS Supervisor test: routed to 'market_data_agent'


### Section 14.1.1 — Market Data Agent

The Market Data Agent retrieves real-time stock information using yfinance. In Simulation Mode, it falls back to `MOCK_STOCK_DATA` which mirrors the exact field structure returned by `ticker.info.get()`.

**Fields retrieved:** Price, Market Cap, P/E Ratio, Day Range, Volume — matching the `get_market_data` function signature from Section 14.1.1 (p.5).


In [3]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 2: Market Data Agent
# Chapter Ref: Section 14.1.1, p.5
# Book: 30 Agents Every AI Engineer Must Build — Imran Ahmad (Packt Publishing)
# ─────────────────────────────────────────────────────────────────────────────

logger.info("Building Market Data Agent (Sec 14.1.1)")

@tool
def get_market_data(symbol: str) -> dict:
    """Fetch current market data for a stock symbol.

    Returns price, market cap, P/E ratio, day range, and volume.
    Falls back to MOCK_STOCK_DATA in Simulation Mode or on API failure.

    Args:
        symbol: Stock ticker symbol (e.g., 'AAPL', 'MSFT', 'GOOGL')

    Chapter Ref: Section 14.1.1, p.5
    Author: Imran Ahmad
    """
    return _get_market_data_impl(symbol)

@graceful_fallback(
    fallback_value=MOCK_STOCK_DATA.get("AAPL", {}),
    section_ref="Sec 14.1.1"
)
def _get_market_data_impl(symbol: str) -> dict:
    """Internal implementation with graceful fallback."""
    # Attempt live data if we might have access
    try:
        import yfinance as yf
        ticker = yf.Ticker(symbol)
        info = ticker.info
        if info and info.get("regularMarketPrice"):
            data = {
                "symbol": symbol,
                "price": info.get("regularMarketPrice", 0),
                "market_cap": info.get("marketCap", 0),
                "pe_ratio": info.get("trailingPE", 0),
                "day_low": info.get("dayLow", 0),
                "day_high": info.get("dayHigh", 0),
                "volume": info.get("volume", 0),
                "fifty_two_week_low": info.get("fiftyTwoWeekLow", 0),
                "fifty_two_week_high": info.get("fiftyTwoWeekHigh", 0),
                "source": "yfinance_live",
            }
            logger.success(f"Live market data retrieved for {symbol}")
            return data
    except Exception:
        pass  # Fall through to mock data

    # Use mock data
    if symbol in MOCK_STOCK_DATA:
        mock = MOCK_STOCK_DATA[symbol]
        data = {
            "symbol": symbol,
            "price": mock["regularMarketPrice"],
            "market_cap": mock["marketCap"],
            "pe_ratio": mock["trailingPE"],
            "day_low": mock["dayLow"],
            "day_high": mock["dayHigh"],
            "volume": mock["volume"],
            "fifty_two_week_low": mock["fiftyTwoWeekLow"],
            "fifty_two_week_high": mock["fiftyTwoWeekHigh"],
            "source": "mock_data (Simulation Mode)",
        }
        logger.warning(f"Using mock market data for {symbol} (Simulation Mode)")
        return data

    return {"symbol": symbol, "error": "No data available", "source": "fallback"}

# ── Build the Market Data Agent using create_react_agent ──
market_tools = [get_market_data]
market_agent = llm.bind_tools(market_tools)

# ── Demo: Fetch market data for portfolio symbols ──
logger.info("Fetching market data for portfolio analysis...")
portfolio_symbols = ["AAPL", "MSFT", "GOOGL"]
market_results = {}

for sym in portfolio_symbols:
    result = get_market_data.invoke({"symbol": sym})
    market_results[sym] = result
    print(f"  {sym}: ${result.get('price', 'N/A'):>10} | "
          f"P/E: {result.get('pe_ratio', 'N/A'):>6} | "
          f"Market Cap: ${result.get('market_cap', 0)/1e9:.0f}B | "
          f"Source: {result.get('source', 'unknown')}")

logger.success(f"Market data collected for {len(market_results)} symbols")


[13:22:14] [Chapter14] INFO Building Market Data Agent (Sec 14.1.1)
[13:22:14] [Chapter14] INFO Fetching market data for portfolio analysis...


[13:22:14] [Chapter14] WARNING Using mock market data for AAPL (Simulation Mode)
  AAPL: $    178.72 | P/E:   28.5 | Market Cap: $2800B | Source: mock_data (Simulation Mode)


[13:22:14] [Chapter14] WARNING Using mock market data for MSFT (Simulation Mode)
  MSFT: $    338.11 | P/E:   32.1 | Market Cap: $2500B | Source: mock_data (Simulation Mode)
[13:22:14] [Chapter14] WARNING Using mock market data for GOOGL (Simulation Mode)
  GOOGL: $     141.8 | P/E:   24.3 | Market Cap: $1780B | Source: mock_data (Simulation Mode)
[13:22:14] [Chapter14] SUCCESS Market data collected for 3 symbols


### Section 14.1.1 — Finnhub Integration: Portfolio Analysis

The `portfolio_analysis()` function extends market data with fundamental metrics from Finnhub's `company_basic_financials` endpoint. Key metrics include P/E ratio, revenue growth, 52-week range, and EPS growth.

In Simulation Mode, `MOCK_FINNHUB_FINANCIALS` provides the exact response structure including `metrics.get('peRatio')`, `revenueGrowth`, `52WeekHigh`, and `52WeekLow` (p.6).


In [4]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 3: Finnhub Integration — Portfolio Analysis
# Chapter Ref: Section 14.1.1, p.6
# Book: 30 Agents Every AI Engineer Must Build — Imran Ahmad (Packt Publishing)
# ─────────────────────────────────────────────────────────────────────────────

logger.info("Building Finnhub Integration (Sec 14.1.1, p.6)")

# ── Conditional Finnhub client setup ──
finnhub_client = None
if config.is_live("FINNHUB_API_KEY"):
    try:
        import finnhub
        finnhub_client = finnhub.Client(api_key=config.get_key("FINNHUB_API_KEY"))
        logger.success("Finnhub client initialized (LIVE mode)")
    except ImportError:
        logger.warning("finnhub-python not installed. Using Simulation Mode.")
else:
    logger.warning("Finnhub API key not provided. Using Simulation Mode.")

@tool
def portfolio_analysis(symbol: str) -> dict:
    """Analyze a stock's fundamental metrics using Finnhub data.

    Retrieves P/E ratio, revenue growth, 52-week high/low, and other
    key financials. Falls back to MOCK_FINNHUB_FINANCIALS in Simulation Mode.

    Args:
        symbol: Stock ticker symbol (e.g., 'AAPL', 'MSFT')

    Chapter Ref: Section 14.1.1, p.6
    Author: Imran Ahmad
    """
    return _portfolio_analysis_impl(symbol)

@graceful_fallback(
    fallback_value={
        "symbol": "UNKNOWN",
        "pe_ratio": "N/A",
        "revenue_growth": "N/A",
        "week_52_high": "N/A",
        "week_52_low": "N/A",
        "source": "fallback",
    },
    section_ref="Sec 14.1.1"
)
def _portfolio_analysis_impl(symbol: str) -> dict:
    """Internal implementation with graceful fallback."""
    metrics = None

    # Attempt live Finnhub data
    if finnhub_client:
        try:
            response = finnhub_client.company_basic_financials(symbol, "all")
            if response and response.get("metric"):
                metrics = response["metric"]
                source = "finnhub_live"
                logger.success(f"Live Finnhub financials retrieved for {symbol}")
        except Exception as e:
            logger.warning(f"Finnhub API error for {symbol}: {e}")

    # Fall back to mock data
    if metrics is None:
        if symbol in MOCK_FINNHUB_FINANCIALS:
            metrics = MOCK_FINNHUB_FINANCIALS[symbol]["metric"]
            source = "mock_data (Simulation Mode)"
            logger.warning(f"Using mock Finnhub financials for {symbol}")
        else:
            return {
                "symbol": symbol,
                "error": "No financial data available",
                "source": "fallback",
            }

    return {
        "symbol": symbol,
        "pe_ratio": metrics.get("peRatio", "N/A"),
        "revenue_growth": metrics.get("revenueGrowth", "N/A"),
        "week_52_high": metrics.get("52WeekHigh", "N/A"),
        "week_52_low": metrics.get("52WeekLow", "N/A"),
        "eps_growth": metrics.get("epsGrowth", "N/A"),
        "dividend_yield": metrics.get("dividendYield", "N/A"),
        "roe": metrics.get("roeTTM", "N/A"),
        "gross_margin": metrics.get("grossMarginTTM", "N/A"),
        "source": source,
    }

# ── Demo: Run portfolio analysis ──
logger.info("Running portfolio analysis for key symbols...")
for sym in ["AAPL", "MSFT"]:
    result = portfolio_analysis.invoke({"symbol": sym})
    print(f"\n  {sym} Fundamentals:")
    print(f"    P/E Ratio:      {result.get('pe_ratio', 'N/A')}")
    print(f"    Revenue Growth:  {result.get('revenue_growth', 'N/A')}")
    print(f"    52-Week High:   ${result.get('week_52_high', 'N/A')}")
    print(f"    52-Week Low:    ${result.get('week_52_low', 'N/A')}")
    print(f"    Source:          {result.get('source', 'unknown')}")

logger.success("Portfolio analysis complete")


[13:22:14] [Chapter14] INFO Building Finnhub Integration (Sec 14.1.1, p.6)
[13:22:14] [Chapter14] WARNING Finnhub API key not provided. Using Simulation Mode.
[13:22:14] [Chapter14] INFO Running portfolio analysis for key symbols...
[13:22:14] [Chapter14] WARNING Using mock Finnhub financials for AAPL

  AAPL Fundamentals:
    P/E Ratio:      28.5
    Revenue Growth:  0.078
    52-Week High:   $182.34
    52-Week Low:    $124.17
    Source:          mock_data (Simulation Mode)
[13:22:14] [Chapter14] WARNING Using mock Finnhub financials for MSFT

  MSFT Fundamentals:
    P/E Ratio:      32.1
    Revenue Growth:  0.124
    52-Week High:   $349.67
    52-Week Low:    $245.61
    Source:          mock_data (Simulation Mode)
[13:22:14] [Chapter14] SUCCESS Portfolio analysis complete


### Section 14.1.1 — Financial News Agent

The Financial News Agent uses Tavily's AI-optimized search to retrieve relevant financial news. In Simulation Mode, `MOCK_TAVILY_NEWS` provides exactly 5 results matching the `TavilySearchResults(max_results=5)` output format (p.7).

The news agent provides market context that the supervisor uses alongside market data and fundamental analysis to inform investment recommendations.


In [5]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 4: Financial News Agent
# Chapter Ref: Section 14.1.1, p.7
# Book: 30 Agents Every AI Engineer Must Build — Imran Ahmad (Packt Publishing)
# ─────────────────────────────────────────────────────────────────────────────

logger.info("Building Financial News Agent (Sec 14.1.1, p.7)")

@tool
def get_financial_news(query: str) -> list:
    """Search for relevant financial news articles.

    Uses Tavily search API for AI-optimized results. Falls back to
    MOCK_TAVILY_NEWS in Simulation Mode, which provides 5 results
    matching the TavilySearchResults output format.

    Args:
        query: Search query for financial news (e.g., 'tech earnings AI investment')

    Chapter Ref: Section 14.1.1, p.7
    Author: Imran Ahmad
    """
    return _get_financial_news_impl(query)

@graceful_fallback(
    fallback_value=MOCK_TAVILY_NEWS,
    section_ref="Sec 14.1.1"
)
def _get_financial_news_impl(query: str) -> list:
    """Internal implementation with graceful fallback."""
    # Attempt live Tavily search
    if config.is_live("TAVILY_API_KEY"):
        try:
            from langchain_community.tools.tavily_search import TavilySearchResults
            tavily_tool = TavilySearchResults(max_results=5)
            results = tavily_tool.invoke({"query": query})
            if results:
                logger.success(f"Live news retrieved: {len(results)} results for '{query}'")
                return results
        except Exception as e:
            logger.warning(f"Tavily API error: {e}")

    # Fall back to mock news data
    logger.warning(f"Using mock news data for '{query}' (Simulation Mode)")
    return MOCK_TAVILY_NEWS

# ── Build the news agent ──
news_tools = [get_financial_news]
news_agent = llm.bind_tools(news_tools)

# ── Demo: Fetch financial news ──
logger.info("Fetching financial news...")
news_results = get_financial_news.invoke({"query": "technology stocks earnings AI investment 2024"})

print(f"\n  Retrieved {len(news_results)} news articles:")
for i, article in enumerate(news_results, 1):
    title = article.get("title", "Untitled")
    score = article.get("score", 0)
    print(f"    {i}. [{score:.2f}] {title}")

logger.success(f"News agent ready with {len(news_results)} articles")


[13:22:14] [Chapter14] INFO Building Financial News Agent (Sec 14.1.1, p.7)
[13:22:14] [Chapter14] INFO Fetching financial news...
[13:22:14] [Chapter14] WARNING Using mock news data for 'technology stocks earnings AI investment 2024' (Simulation Mode)

  Retrieved 5 news articles:
    1. [0.97] Federal Reserve Holds Interest Rates Steady, Signals Patience on Cuts
    2. [0.94] Tech Earnings Season: Mixed Results Highlight AI Investment Surge
    3. [0.91] S&P 500 Reaches New All-Time High on Strong Employment Data
    4. [0.88] Global Supply Chain Recovery Eases Inflation Pressures
    5. [0.85] Semiconductor Sector Rallies on Data Center Demand Forecast
[13:22:14] [Chapter14] SUCCESS News agent ready with 5 articles


### Section 14.1 — StateGraph Assembly (p.7-9)

The Financial Advisory Agent is assembled as a LangGraph `StateGraph` where:
- Each specialized agent is a **node** in the graph
- The **supervisor** provides conditional routing between nodes
- A `messages` list in the shared state accumulates results from each agent
- The graph streams execution, printing each node's output as it completes

This architecture ensures that agents execute in a coordinated sequence rather than independently, with the supervisor maintaining full control of the workflow.


In [6]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 5: StateGraph Assembly
# Chapter Ref: Section 14.1, p.7-9
# Book: 30 Agents Every AI Engineer Must Build — Imran Ahmad (Packt Publishing)
# ─────────────────────────────────────────────────────────────────────────────

logger.info("Assembling Financial Advisory StateGraph (Sec 14.1, p.7-9)")

# ── Define the shared agent state ──
class AgentState(TypedDict):
    """Shared state passed between all nodes in the financial advisory graph.

    Chapter Ref: Section 14.1, p.7
    Author: Imran Ahmad
    """
    messages: Annotated[list, lambda x, y: x + y]
    next_agent: str
    market_data: dict
    analysis_data: dict
    news_data: list
    iteration_count: int

# ── Agent node wrapper ──
@graceful_fallback(fallback_value={}, section_ref="Sec 14.1, p.8")
def agent_node(state: dict, agent_name: str, agent_fn) -> dict:
    """Generic agent node wrapper for the StateGraph.

    Executes the given agent function, logs execution, and returns
    updated state. Wrapped in @graceful_fallback to ensure the graph
    never crashes.

    Chapter Ref: Section 14.1, p.8
    Author: Imran Ahmad
    """
    logger.info(f"Executing agent node: {agent_name}")
    result = agent_fn(state)
    logger.success(f"Agent node '{agent_name}' completed")
    return result

# ── Define individual agent functions ──
def market_data_node(state: dict) -> dict:
    """Market Data Agent node — fetches data for portfolio symbols.

    Chapter Ref: Section 14.1.1
    Author: Imran Ahmad
    """
    symbols = ["AAPL", "MSFT", "GOOGL"]
    data = {}
    for sym in symbols:
        result = get_market_data.invoke({"symbol": sym})
        data[sym] = result
    return {
        "messages": [AIMessage(content=f"Market data collected for {', '.join(symbols)}")],
        "market_data": data,
        "next_agent": "analysis_agent",
    }

def analysis_node(state: dict) -> dict:
    """Analysis Agent node — runs portfolio analysis on available symbols.

    Chapter Ref: Section 14.1.1, p.6
    Author: Imran Ahmad
    """
    analysis = {}
    for sym in ["AAPL", "MSFT"]:
        result = portfolio_analysis.invoke({"symbol": sym})
        analysis[sym] = result
    return {
        "messages": [AIMessage(content="Portfolio analysis complete for AAPL, MSFT")],
        "analysis_data": analysis,
        "next_agent": "news_agent",
    }

def news_node(state: dict) -> dict:
    """News Agent node — retrieves relevant financial news.

    Chapter Ref: Section 14.1.1, p.7
    Author: Imran Ahmad
    """
    results = get_financial_news.invoke(
        {"query": "technology stocks market outlook investment strategy"}
    )
    return {
        "messages": [AIMessage(content=f"Retrieved {len(results)} news articles")],
        "news_data": results,
        "next_agent": "FINISH",
    }

# ── Routing function for conditional edges ──
def route_supervisor(state: dict) -> str:
    """Determine the next node based on supervisor routing.

    Chapter Ref: Section 14.1, p.9
    Author: Imran Ahmad
    """
    next_agent = state.get("next_agent", "FINISH")
    if next_agent == "FINISH":
        return "end"
    return next_agent

# ── Build the StateGraph ──
from langgraph.graph import StateGraph, END

workflow = StateGraph(AgentState)

# Add nodes
workflow.add_node("market_data_agent", market_data_node)
workflow.add_node("analysis_agent", analysis_node)
workflow.add_node("news_agent", news_node)

# Set entry point
workflow.set_entry_point("market_data_agent")

# Add conditional edges
workflow.add_conditional_edges(
    "market_data_agent",
    route_supervisor,
    {
        "analysis_agent": "analysis_agent",
        "end": END,
    }
)
workflow.add_conditional_edges(
    "analysis_agent",
    route_supervisor,
    {
        "news_agent": "news_agent",
        "end": END,
    }
)
workflow.add_conditional_edges(
    "news_agent",
    route_supervisor,
    {
        "end": END,
    }
)

# Compile the graph
financial_graph = workflow.compile()
logger.success("Financial Advisory StateGraph compiled successfully")

# ── Execute the graph with streaming ──
logger.info("Running Financial Advisory Agent (streaming execution)...")
initial_state = {
    "messages": [HumanMessage(content="Analyze AAPL, MSFT, GOOGL for a moderate growth portfolio")],
    "next_agent": "market_data_agent",
    "market_data": {},
    "analysis_data": {},
    "news_data": [],
    "iteration_count": 0,
}

print("\n" + "─" * 60)
print("  FINANCIAL ADVISORY AGENT — Streaming Execution")
print("─" * 60)

final_state = None
for step in financial_graph.stream(initial_state):
    for node_name, node_output in step.items():
        print(f"\n  ▸ Node: {node_name}")
        if isinstance(node_output, dict):
            msgs = node_output.get("messages", [])
            for msg in msgs:
                if hasattr(msg, "content"):
                    print(f"    Result: {msg.content}")
        final_state = node_output

print("\n" + "─" * 60)
logger.success("Financial Advisory Agent execution complete")


[13:22:14] [Chapter14] INFO Assembling Financial Advisory StateGraph (Sec 14.1, p.7-9)


[13:22:14] [Chapter14] SUCCESS Financial Advisory StateGraph compiled successfully
[13:22:14] [Chapter14] INFO Running Financial Advisory Agent (streaming execution)...

────────────────────────────────────────────────────────────
  FINANCIAL ADVISORY AGENT — Streaming Execution
────────────────────────────────────────────────────────────


[13:22:14] [Chapter14] WARNING Using mock market data for AAPL (Simulation Mode)
[13:22:14] [Chapter14] WARNING Using mock market data for MSFT (Simulation Mode)
[13:22:14] [Chapter14] WARNING Using mock market data for GOOGL (Simulation Mode)

  ▸ Node: market_data_agent
    Result: Market data collected for AAPL, MSFT, GOOGL
[13:22:14] [Chapter14] WARNING Using mock Finnhub financials for AAPL
[13:22:14] [Chapter14] WARNING Using mock Finnhub financials for MSFT

  ▸ Node: analysis_agent
    Result: Portfolio analysis complete for AAPL, MSFT
[13:22:14] [Chapter14] WARNING Using mock news data for 'technology stocks market outlook investment strategy' (Simulation Mode)

  ▸ Node: news_agent
    Result: Retrieved 5 news articles

────────────────────────────────────────────────────────────
[13:22:14] [Chapter14] SUCCESS Financial Advisory Agent execution complete


### Section 14.1.2 — Risk Assessment (p.10-14)

The `RiskScorer` class implements quantitative risk metrics:

- **Value at Risk (VaR)** at the 95th percentile — the maximum expected loss on 95% of trading days
- **Conditional VaR (CVaR)** — the expected loss in the worst 5% of scenarios (tail risk)
- **Annualized Volatility** — standard deviation of returns scaled to annual frequency
- **Maximum Drawdown** — largest peak-to-trough decline over the observation period

The risk score combines these metrics into a single 0-10 rating, which is then adjusted based on the client's risk tolerance to produce a suitability assessment.

Finnhub quote data (`dp` field) provides an additional price-change-based risk classification:
- `abs(dp) < 2` → Low Risk
- `2 ≤ abs(dp) < 5` → Moderate Risk
- `abs(dp) ≥ 5` → High Risk


In [7]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 6: Risk Assessment
# Chapter Ref: Section 14.1.2, p.10-14
# Book: 30 Agents Every AI Engineer Must Build — Imran Ahmad (Packt Publishing)
# ─────────────────────────────────────────────────────────────────────────────

logger.info("Building Risk Assessment System (Sec 14.1.2, p.10-14)")

class RiskScorer:
    """Quantitative risk scoring engine.

    Computes VaR, CVaR, annualized volatility, maximum drawdown, and an
    aggregate risk score (0-10) from historical price data.

    Chapter Ref: Section 14.1.2, p.10-14
    Author: Imran Ahmad
    """

    def __init__(self, confidence_level: float = 0.95):
        self.confidence_level = confidence_level

    @graceful_fallback(
        fallback_value={"risk_score": 5.0, "category": "Moderate", "source": "fallback"},
        section_ref="Sec 14.1.2"
    )
    def compute_risk_score(self, symbol: str, price_history: dict = None) -> dict:
        """Compute comprehensive risk metrics for a symbol.

        Args:
            symbol: Stock ticker symbol
            price_history: Dict with 'Close' key containing price list.
                           If None, uses generate_mock_price_history().

        Returns:
            Dict with VaR, CVaR, volatility, max_drawdown, risk_score, category.

        Chapter Ref: Section 14.1.2, p.11-12
        Author: Imran Ahmad
        """
        # Get price history
        if price_history is None:
            price_history = generate_mock_price_history(symbol, days=90)

        closes = np.array(price_history["Close"])
        returns = np.diff(closes) / closes[:-1]

        # VaR at confidence level (Sec 14.1.2, p.11)
        var_percentile = (1 - self.confidence_level) * 100
        var = float(np.percentile(returns, var_percentile))

        # CVaR — expected shortfall beyond VaR (Sec 14.1.2, p.11)
        cvar = float(np.mean(returns[returns <= var]))

        # Annualized volatility (Sec 14.1.2, p.12)
        daily_vol = float(np.std(returns))
        annualized_vol = daily_vol * np.sqrt(252)

        # Maximum drawdown (Sec 14.1.2, p.12)
        cumulative = np.cumprod(1 + returns)
        running_max = np.maximum.accumulate(cumulative)
        drawdowns = (cumulative - running_max) / running_max
        max_drawdown = float(np.min(drawdowns))

        # Aggregate risk score (0-10 scale) (Sec 14.1.2, p.13)
        # Higher volatility and deeper drawdowns → higher score
        vol_score = min(annualized_vol / 0.05, 10)  # 50% vol = max score
        dd_score = min(abs(max_drawdown) / 0.03, 10)  # 30% drawdown = max score
        var_score = min(abs(var) / 0.005, 10)  # 5% daily VaR = max score
        risk_score = round((vol_score * 0.4 + dd_score * 0.3 + var_score * 0.3), 1)

        # Category classification
        if risk_score < 3.5:
            category = "Low"
        elif risk_score < 6.5:
            category = "Moderate"
        else:
            category = "High"

        return {
            "symbol": symbol,
            "var_95": round(var, 6),
            "cvar_95": round(cvar, 6),
            "annualized_volatility": round(annualized_vol, 4),
            "max_drawdown": round(max_drawdown, 4),
            "risk_score": risk_score,
            "category": category,
            "confidence_level": self.confidence_level,
            "observation_days": len(closes),
        }

    @graceful_fallback(
        fallback_value="Low Risk",
        section_ref="Sec 14.1.2"
    )
    def classify_by_price_change(self, symbol: str) -> str:
        """Classify risk using Finnhub quote dp (percent change) field.

        Classification rules (Sec 14.1.2, p.10):
            abs(dp) < 2    → "Low Risk"
            2 ≤ abs(dp) < 5 → "Moderate Risk"
            abs(dp) ≥ 5    → "High Risk"

        Author: Imran Ahmad
        """
        quote = MOCK_FINNHUB_QUOTES.get(symbol, {})
        dp = abs(quote.get("dp", 0))

        if dp < 2:
            return "Low Risk"
        elif dp < 5:
            return "Moderate Risk"
        else:
            return "High Risk"


@graceful_fallback(
    fallback_value={"suitability": "Unknown", "recommendation": "Consult advisor"},
    section_ref="Sec 14.1.2"
)
def assess_risk(portfolio_scores: list, client_tolerance: str) -> dict:
    """Assess overall portfolio risk against client tolerance.

    Adjusts the raw risk score based on the client's stated tolerance
    and produces a suitability determination.

    Args:
        portfolio_scores: List of risk score dicts from RiskScorer
        client_tolerance: One of 'conservative', 'moderate', 'aggressive'

    Chapter Ref: Section 14.1.2, p.13-14
    Author: Imran Ahmad
    """
    if not portfolio_scores:
        return {"suitability": "Unknown", "recommendation": "No data available"}

    # Average portfolio risk score
    avg_score = np.mean([s.get("risk_score", 5.0) for s in portfolio_scores])

    # Tolerance thresholds
    tolerance_thresholds = {
        "conservative": 4.0,
        "moderate": 6.5,
        "aggressive": 8.5,
    }
    threshold = tolerance_thresholds.get(client_tolerance, 6.5)

    if avg_score <= threshold:
        suitability = "SUITABLE"
        recommendation = (
            f"Portfolio risk score ({avg_score:.1f}) is within acceptable "
            f"bounds for a {client_tolerance} investor (threshold: {threshold})."
        )
    else:
        suitability = "EXCEEDS_TOLERANCE"
        recommendation = (
            f"Portfolio risk score ({avg_score:.1f}) exceeds the threshold "
            f"for a {client_tolerance} investor (threshold: {threshold}). "
            f"Consider reducing equity allocation or adding more fixed income."
        )

    return {
        "avg_risk_score": round(float(avg_score), 1),
        "client_tolerance": client_tolerance,
        "threshold": threshold,
        "suitability": suitability,
        "recommendation": recommendation,
    }


# ── Demo: Risk Assessment ──
risk_scorer = RiskScorer(confidence_level=0.95)

print("\n" + "═" * 60)
print("  RISK ASSESSMENT — Section 14.1.2")
print("═" * 60)

portfolio_risk_scores = []
for sym in ["AAPL", "MSFT", "GOOGL"]:
    score = risk_scorer.compute_risk_score(sym)
    portfolio_risk_scores.append(score)
    dp_class = risk_scorer.classify_by_price_change(sym)
    print(f"\n  {sym}:")
    print(f"    VaR (95%):             {score['var_95']:+.4%} daily")
    print(f"    CVaR (95%):            {score['cvar_95']:+.4%} daily")
    print(f"    Annualized Volatility: {score['annualized_volatility']:.2%}")
    print(f"    Max Drawdown:          {score['max_drawdown']:.2%}")
    print(f"    Risk Score:            {score['risk_score']}/10 ({score['category']})")
    print(f"    Price Change Class:    {dp_class}")

# Assess against moderate client
suitability = assess_risk(portfolio_risk_scores, "moderate")
print(f"\n{'─' * 60}")
print(f"  Portfolio Suitability Assessment:")
print(f"    Client Tolerance:  {suitability['client_tolerance']}")
print(f"    Avg Risk Score:    {suitability['avg_risk_score']}/10")
print(f"    Threshold:         {suitability['threshold']}")
print(f"    Suitability:       {suitability['suitability']}")
print(f"    Recommendation:    {suitability['recommendation']}")
print("═" * 60)

logger.success("Risk Assessment system operational")


[13:22:14] [Chapter14] INFO Building Risk Assessment System (Sec 14.1.2, p.10-14)

════════════════════════════════════════════════════════════
  RISK ASSESSMENT — Section 14.1.2
════════════════════════════════════════════════════════════



  AAPL:
    VaR (95%):             -2.0739% daily
    CVaR (95%):            -2.6001% daily
    Annualized Volatility: 22.11%
    Max Drawdown:          -11.59%
    Risk Score:            4.2/10 (Moderate)
    Price Change Class:    Low Risk

  MSFT:
    VaR (95%):             -2.1082% daily
    CVaR (95%):            -2.5715% daily
    Annualized Volatility: 19.34%
    Max Drawdown:          -19.19%
    Risk Score:            4.7/10 (Moderate)
    Price Change Class:    Low Risk

  GOOGL:
    VaR (95%):             -2.4683% daily
    CVaR (95%):            -3.3192% daily
    Annualized Volatility: 26.01%
    Max Drawdown:          -11.61%
    Risk Score:            4.7/10 (Moderate)
    Price Change Class:    Moderate Risk

────────────────────────────────────────────────────────────
  Portfolio Suitability Assessment:
    Client Tolerance:  moderate
    Avg Risk Score:    4.5/10
    Threshold:         6.5
    Suitability:       SUITABLE
    Recommendation:    Portfolio risk score (4

### Section 14.1.3 — Personalized Financial Planning & Compliance Gate (p.14-18)

The personalized planning system consists of:

1. **ClientProfileAgent** — Generates investment plans tailored to client profiles (risk tolerance, horizon, constraints)
2. **AdvisoryState** — Shared state for the compliance workflow
3. **validate_compliance()** — Structural validation gate that checks suitability, concentration limits, and disclosure requirements
4. **route_after_compliance()** — Routes approved plans to delivery, or rejected plans back for revision

This implements the **compliance-by-architecture** pattern: validation is structural, not advisory. Non-compliant plans are automatically rejected and revised rather than simply flagged with warnings.

> **Note:** This architectural approach directly addresses the lesson from the Knight Capital incident — compliance checks must be mandatory workflow gates, not optional review steps.


In [8]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 7: Personalized Planning & Compliance Gate
# Chapter Ref: Section 14.1.3, p.14-18
# Book: 30 Agents Every AI Engineer Must Build — Imran Ahmad (Packt Publishing)
# ─────────────────────────────────────────────────────────────────────────────

logger.info("Building Personalized Planning & Compliance Gate (Sec 14.1.3, p.14-18)")

# ── Advisory State for the compliance workflow ──
class AdvisoryState(TypedDict):
    """Shared state for the advisory compliance workflow.

    Chapter Ref: Section 14.1.3, p.14
    Author: Imran Ahmad
    """
    client_profile: dict
    investment_plan: dict
    risk_assessment: dict
    compliance_status: str
    compliance_issues: list
    revision_count: int
    messages: list

# ── Client Profile Agent ──
class ClientProfileAgent:
    """Generates personalized investment plans based on client profiles.

    Uses the client's risk tolerance, investment horizon, goals, and
    constraints to produce a tailored allocation recommendation.

    Chapter Ref: Section 14.1.3, p.15-16
    Author: Imran Ahmad
    """

    # Allocation templates by risk tolerance (Sec 14.1.3, p.16)
    ALLOCATION_TEMPLATES = {
        "conservative": {
            "us_equities": 0.25,
            "international_equities": 0.10,
            "fixed_income": 0.50,
            "alternatives": 0.15,
        },
        "moderate": {
            "us_equities": 0.45,
            "international_equities": 0.20,
            "fixed_income": 0.25,
            "alternatives": 0.10,
        },
        "aggressive": {
            "us_equities": 0.60,
            "international_equities": 0.25,
            "fixed_income": 0.10,
            "alternatives": 0.05,
        },
    }

    @graceful_fallback(
        fallback_value={
            "allocation": {"us_equities": 0.45, "international_equities": 0.20,
                           "fixed_income": 0.25, "alternatives": 0.10},
            "source": "fallback",
        },
        section_ref="Sec 14.1.3"
    )
    def generate_plan(self, client_profile: dict, risk_assessment: dict) -> dict:
        """Generate an investment plan for the given client profile.

        Chapter Ref: Section 14.1.3, p.15-17
        Author: Imran Ahmad
        """
        tolerance = client_profile.get("risk_tolerance", "moderate")
        investment = client_profile.get("initial_investment", 50000)
        horizon = client_profile.get("investment_horizon", "10 years")

        allocation = self.ALLOCATION_TEMPLATES.get(tolerance, self.ALLOCATION_TEMPLATES["moderate"])

        # Calculate dollar amounts
        allocation_details = {}
        for asset_class, pct in allocation.items():
            allocation_details[asset_class] = {
                "percentage": pct,
                "amount": round(investment * pct, 2),
            }

        plan = {
            "client_id": client_profile.get("client_id", "unknown"),
            "risk_tolerance": tolerance,
            "investment_horizon": horizon,
            "initial_investment": investment,
            "allocation": allocation,
            "allocation_details": allocation_details,
            "risk_score": risk_assessment.get("avg_risk_score", 5.0),
            "suitability": risk_assessment.get("suitability", "UNKNOWN"),
        }

        logger.success(f"Investment plan generated for client {plan['client_id']}")
        return plan


# ── Compliance Validation Gate ──
@graceful_fallback(
    fallback_value={"status": "ERROR", "issues": ["Compliance check failed"]},
    section_ref="Sec 14.1.3"
)
def validate_compliance(state: dict) -> dict:
    """Structural compliance validation gate.

    Checks:
    1. Suitability — Risk level matches client tolerance
    2. Concentration — No single asset class exceeds 60%
    3. Disclosure — Required disclaimers present

    This is a mandatory workflow gate, not an advisory step.

    Chapter Ref: Section 14.1.3, p.17-18
    Author: Imran Ahmad
    """
    plan = state.get("investment_plan", {})
    profile = state.get("client_profile", {})
    issues = []

    # Check 1: Suitability
    suitability = plan.get("suitability", "UNKNOWN")
    if suitability == "EXCEEDS_TOLERANCE":
        issues.append(
            f"FAIL: Risk score exceeds {profile.get('risk_tolerance', 'unknown')} tolerance threshold"
        )

    # Check 2: Concentration limits
    allocation = plan.get("allocation", {})
    for asset_class, pct in allocation.items():
        if pct > 0.60:
            issues.append(
                f"FAIL: {asset_class} allocation ({pct:.0%}) exceeds 60% concentration limit"
            )

    # Check 3: Client constraints
    constraints = profile.get("constraints", [])
    for constraint in constraints:
        if "maximum 40% in any single sector" in constraint.lower():
            for asset_class, pct in allocation.items():
                if pct > 0.40 and "equities" not in asset_class:
                    issues.append(f"WARN: Review constraint — '{constraint}'")

    if issues:
        status = "REJECTED"
        logger.error(f"Compliance REJECTED: {len(issues)} issue(s) found")
    else:
        status = "APPROVED"
        logger.success("Compliance APPROVED — all checks passed")

    return {
        "compliance_status": status,
        "compliance_issues": issues,
        "messages": state.get("messages", []) + [
            AIMessage(content=f"Compliance validation: {status}. Issues: {len(issues)}")
        ],
    }


def route_after_compliance(state: dict) -> str:
    """Route based on compliance result — approve or revise.

    Chapter Ref: Section 14.1.3, p.18
    Author: Imran Ahmad
    """
    if state.get("compliance_status") == "APPROVED":
        return "deliver"
    elif state.get("revision_count", 0) >= 2:
        logger.warning("Maximum revision attempts reached. Escalating to human review.")
        return "deliver"
    else:
        return "revise"


def revise_plan(state: dict) -> dict:
    """Revise a rejected plan by adjusting allocation toward constraints.

    Chapter Ref: Section 14.1.3, p.18
    Author: Imran Ahmad
    """
    logger.warning("Revising investment plan based on compliance feedback...")
    plan = state.get("investment_plan", {})

    # Simple revision: shift toward more conservative allocation
    revised_allocation = {
        "us_equities": 0.35,
        "international_equities": 0.15,
        "fixed_income": 0.35,
        "alternatives": 0.15,
    }
    plan["allocation"] = revised_allocation
    plan["revision_note"] = "Allocation adjusted toward conservative profile per compliance feedback"

    return {
        "investment_plan": plan,
        "revision_count": state.get("revision_count", 0) + 1,
        "messages": state.get("messages", []) + [
            AIMessage(content="Plan revised: shifted allocation toward conservative profile")
        ],
    }


def deliver_plan(state: dict) -> dict:
    """Deliver the approved investment plan.

    Chapter Ref: Section 14.1.3, p.18
    Author: Imran Ahmad
    """
    status = state.get("compliance_status", "UNKNOWN")
    logger.success(f"Delivering investment plan (compliance: {status})")
    return {
        "messages": state.get("messages", []) + [
            AIMessage(content=f"Investment plan delivered. Compliance status: {status}")
        ],
    }


# ── Build the Compliance StateGraph ──
compliance_workflow = StateGraph(AdvisoryState)

compliance_workflow.add_node("validate", validate_compliance)
compliance_workflow.add_node("revise", revise_plan)
compliance_workflow.add_node("deliver", deliver_plan)

compliance_workflow.set_entry_point("validate")
compliance_workflow.add_conditional_edges(
    "validate",
    route_after_compliance,
    {
        "deliver": "deliver",
        "revise": "revise",
    }
)
compliance_workflow.add_edge("revise", "validate")
compliance_workflow.add_edge("deliver", END)

compliance_graph = compliance_workflow.compile()
logger.success("Compliance StateGraph compiled with revise loop")

# ── Demo: Test with moderate client ──
profile_agent = ClientProfileAgent()
client = MOCK_CLIENT_PROFILES["retail_client_4521"]
plan = profile_agent.generate_plan(client, suitability)

print("\n" + "═" * 60)
print("  PERSONALIZED PLANNING — Section 14.1.3")
print("═" * 60)
print(f"  Client: {client['name']} ({client['client_id']})")
print(f"  Investment: ${client['initial_investment']:,.0f}")
print(f"  Tolerance: {client['risk_tolerance']}")
print(f"  Horizon: {client['investment_horizon']}")
print(f"\n  Recommended Allocation:")
for asset_class, details in plan.get("allocation_details", {}).items():
    label = asset_class.replace("_", " ").title()
    print(f"    {label:30s} {details['percentage']:6.0%}  (${details['amount']:>10,.2f})")

# Run compliance check
compliance_state = {
    "client_profile": client,
    "investment_plan": plan,
    "risk_assessment": suitability,
    "compliance_status": "",
    "compliance_issues": [],
    "revision_count": 0,
    "messages": [],
}

print(f"\n{'─' * 60}")
print("  Compliance Validation:")
for step in compliance_graph.stream(compliance_state):
    for node_name, output in step.items():
        status = output.get("compliance_status", "")
        if status:
            print(f"    ▸ {node_name}: {status}")
            for issue in output.get("compliance_issues", []):
                print(f"      - {issue}")

print("═" * 60)
logger.success("Personalized Planning & Compliance system operational")


[13:22:14] [Chapter14] INFO Building Personalized Planning & Compliance Gate (Sec 14.1.3, p.14-18)
[13:22:14] [Chapter14] SUCCESS Compliance StateGraph compiled with revise loop
[13:22:14] [Chapter14] SUCCESS Investment plan generated for client retail_client_4521

════════════════════════════════════════════════════════════
  PERSONALIZED PLANNING — Section 14.1.3
════════════════════════════════════════════════════════════
  Client: Sarah Chen (retail_client_4521)
  Investment: $50,000
  Tolerance: moderate
  Horizon: 10 years

  Recommended Allocation:
    Us Equities                       45%  ($ 22,500.00)
    International Equities            20%  ($ 10,000.00)
    Fixed Income                      25%  ($ 12,500.00)
    Alternatives                      10%  ($  5,000.00)

────────────────────────────────────────────────────────────
  Compliance Validation:
[13:22:14] [Chapter14] SUCCESS Compliance APPROVED — all checks passed


    ▸ validate: APPROVED
[13:22:14] [Chapter14] SUCCESS Delivering investment plan (compliance: APPROVED)
════════════════════════════════════════════════════════════
[13:22:14] [Chapter14] SUCCESS Personalized Planning & Compliance system operational


### Section 14.1.4 — RetailAdvisor Case Study (p.18-20)

This cell demonstrates the complete end-to-end financial advisory workflow:

1. A retail client submits: "$50,000 to invest, moderate growth, 10-year horizon"
2. The supervisor orchestrates market data collection, analysis, and news retrieval
3. Risk scoring produces VaR/CVaR metrics and a suitability assessment
4. The personalized plan generates a 45/20/25/10 allocation (matching the chapter's inter-agent message protocol on p.19)
5. The compliance gate validates and approves the plan

The inter-agent JSON message format matches the exact protocol described in the chapter.


In [9]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 8: RetailAdvisor Case Study
# Chapter Ref: Section 14.1.4, p.18-20
# Book: 30 Agents Every AI Engineer Must Build — Imran Ahmad (Packt Publishing)
# ─────────────────────────────────────────────────────────────────────────────

logger.info("Running RetailAdvisor Case Study (Sec 14.1.4, p.18-20)")

print("\n" + "█" * 60)
print("  RETAIL ADVISOR CASE STUDY — Section 14.1.4")
print("  Client Query: '$50K moderate growth 10yr'")
print("█" * 60)

# ── Step 1: Client Profile ──
client = MOCK_CLIENT_PROFILES["retail_client_4521"]
print(f"\n  ┌─ CLIENT PROFILE ─────────────────────────────────────┐")
print(f"  │ Name:       {client['name']:40s}│")
print(f"  │ ID:         {client['client_id']:40s}│")
print(f"  │ Investment: ${client['initial_investment']:>38,}│")
print(f"  │ Goal:       {client['investment_goal']:40s}│")
print(f"  │ Tolerance:  {client['risk_tolerance']:40s}│")
print(f"  │ Horizon:    {client['investment_horizon']:40s}│")
print(f"  └────────────────────────────────────────────────────────┘")

# ── Step 2: Market Data Collection ──
print(f"\n  ┌─ STEP 1: MARKET DATA COLLECTION ────────────────────┐")
for sym in ["AAPL", "MSFT", "GOOGL"]:
    data = market_results.get(sym, MOCK_STOCK_DATA.get(sym, {}))
    price = data.get("price", data.get("regularMarketPrice", "N/A"))
    pe = data.get("pe_ratio", data.get("trailingPE", "N/A"))
    print(f"  │ {sym:6s} Price: ${price:>10} | P/E: {pe:>6}             │")
print(f"  └────────────────────────────────────────────────────────┘")

# ── Step 3: Risk Assessment ──
print(f"\n  ┌─ STEP 2: RISK ASSESSMENT ────────────────────────────┐")
for score in portfolio_risk_scores:
    sym = score["symbol"]
    print(f"  │ {sym:6s} Risk Score: {score['risk_score']:>4}/10 ({score['category']:>8s}) "
          f"VaR: {score['var_95']:+.4f}   │")
print(f"  │{'─' * 56}│")
print(f"  │ Suitability: {suitability['suitability']:42s}│")
print(f"  └────────────────────────────────────────────────────────┘")

# ── Step 4: Inter-Agent Message (matching chapter protocol, p.19) ──
print(f"\n  ┌─ STEP 3: INTER-AGENT MESSAGE PROTOCOL (p.19) ───────┐")
msg = MOCK_INTER_AGENT_MESSAGE
alloc = msg["payload"]["recommended_allocation"]
print(f"  │ Source:  {msg['source_agent']:45s}│")
print(f"  │ Target:  {msg['target_agent']:45s}│")
print(f"  │ Status:  {msg['status']:45s}│")
print(f"  │{'─' * 56}│")
print(f"  │ Allocation:                                            │")
print(f"  │   US Equities:          {alloc['us_equities']:>5.0%}  (${alloc['us_equities'] * 50000:>10,.0f})     │")
print(f"  │   International Eq.:    {alloc['international_equities']:>5.0%}  (${alloc['international_equities'] * 50000:>10,.0f})     │")
print(f"  │   Fixed Income:         {alloc['fixed_income']:>5.0%}  (${alloc['fixed_income'] * 50000:>10,.0f})     │")
print(f"  │   Alternatives:         {alloc['alternatives']:>5.0%}  (${alloc['alternatives'] * 50000:>10,.0f})     │")
print(f"  └────────────────────────────────────────────────────────┘")

# ── Step 5: JSON Message Dump ──
print(f"\n  Inter-Agent JSON Message:")
print(f"  {json.dumps(msg['payload']['recommended_allocation'], indent=4)}")

# ── Step 6: Compliance Validation ──
print(f"\n  ┌─ STEP 4: COMPLIANCE VALIDATION ──────────────────────┐")
checklist = msg["payload"]["compliance_checklist"]
for check, passed in checklist.items():
    label = check.replace("_", " ").title()
    symbol = "✓" if passed else "✗"
    status_str = "PASS" if passed else "FAIL"
    print(f"  │   {symbol} {label:40s} {status_str:>6s}   │")
print(f"  │{'─' * 56}│")
print(f"  │   Overall: APPROVED — Plan cleared for delivery        │")
print(f"  └────────────────────────────────────────────────────────┘")

print(f"\n{'█' * 60}")
logger.success("RetailAdvisor Case Study complete — all steps executed successfully")
logger.success("Financial Advisory Agent (Section 14.1) COMPLETE")


[13:22:14] [Chapter14] INFO Running RetailAdvisor Case Study (Sec 14.1.4, p.18-20)

████████████████████████████████████████████████████████████
  RETAIL ADVISOR CASE STUDY — Section 14.1.4
  Client Query: '$50K moderate growth 10yr'
████████████████████████████████████████████████████████████

  ┌─ CLIENT PROFILE ─────────────────────────────────────┐
  │ Name:       Sarah Chen                              │
  │ ID:         retail_client_4521                      │
  │ Investment: $                                50,000│
  │ Goal:       moderate growth                         │
  │ Tolerance:  moderate                                │
  │ Horizon:    10 years                                │
  └────────────────────────────────────────────────────────┘

  ┌─ STEP 1: MARKET DATA COLLECTION ────────────────────┐
  │ AAPL   Price: $    178.72 | P/E:   28.5             │
  │ MSFT   Price: $    338.11 | P/E:   32.1             │
  │ GOOGL  Price: $     141.8 | P/E:   24.3             │
  └─

---

## Section 14.2 — Legal Intelligence Agent

### Section 14.2.1 — Legal Knowledge Base (p.20-23)

The Legal Intelligence Agent is built on a **hybrid retrieval** architecture that combines keyword matching with semantic similarity search. The `LegalKnowledgeBase` class:

1. **Ingests cases** — Each case is stored with full metadata (court, jurisdiction, authority level, status) in the `MockVectorStore`
2. **Hybrid search** — Combines keyword-based filtering with vector similarity to achieve higher recall than either method alone
3. **Authority-weighted ranking** — Results are re-ranked by court hierarchy (`authority_level` 0-10), so Supreme Court rulings rank above district court opinions regardless of semantic similarity

This is powered by `MockEmbeddingModel` (hash-based deterministic embeddings) and `MockVectorStore` (in-memory cosine similarity with metadata filtering).

> **Note (Schwartz v. Avianca Incident):** Section 14.2 highlights the 2023 incident where attorneys submitted fabricated case citations from an AI chatbot to a federal court. This underscores why the citation verification pipeline (Section 14.2.4) is a critical component of any legal AI system. Our knowledge base includes one fabricated case (`Varghese v. Tech Corp International`) specifically designed to test this verification.


In [10]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 9: Legal Knowledge Base
# Chapter Ref: Section 14.2.1, p.20-23
# Book: 30 Agents Every AI Engineer Must Build — Imran Ahmad (Packt Publishing)
# ─────────────────────────────────────────────────────────────────────────────

logger.info("Building Legal Knowledge Base (Sec 14.2.1, p.20-23)")

class LegalKnowledgeBase:
    """Hybrid retrieval legal knowledge base with authority-weighted ranking.

    Combines keyword filtering with semantic vector search, then re-ranks
    results by court authority level to ensure that higher-court rulings
    are prioritized.

    Chapter Ref: Section 14.2.1, p.20-23
    Author: Imran Ahmad
    """

    def __init__(self):
        self.embedding_model = MockEmbeddingModel(dimension=384)
        self.vector_store = MockVectorStore(embedding_model=self.embedding_model)
        self.case_metadata = {}  # case_id → full metadata

    @graceful_fallback(
        fallback_value=None,
        section_ref="Sec 14.2.1"
    )
    def ingest_case(self, case: dict) -> None:
        """Ingest a legal case into the knowledge base.

        Stores the case summary as a vector embedding with full metadata
        for later retrieval and filtering.

        Args:
            case: Dict with case_name, citation, court, jurisdiction,
                  authority_level, status, summary, holding, key_issues.

        Chapter Ref: Section 14.2.1, p.21
        Author: Imran Ahmad
        """
        case_id = case["case_id"]
        # Build searchable text from summary + holding + key issues
        text_parts = [
            case.get("summary", ""),
            case.get("holding", ""),
            " ".join(case.get("key_issues", [])),
        ]
        searchable_text = " ".join(text_parts)

        metadata = {
            "case_name": case["case_name"],
            "citation": case["citation"],
            "court": case["court"],
            "jurisdiction": case["jurisdiction"],
            "authority_level": case["authority_level"],
            "year": case.get("year", 0),
            "status": case["status"],
            "key_issues": case.get("key_issues", []),
        }

        self.vector_store.upsert(
            doc_id=case_id,
            text=searchable_text,
            metadata=metadata,
        )
        self.case_metadata[case_id] = case
        logger.info(f"Ingested case: {case['case_name'][:50]} (authority: {case['authority_level']})")

    @graceful_fallback(
        fallback_value=[],
        section_ref="Sec 14.2.1"
    )
    def hybrid_search(
        self,
        query: str,
        top_k: int = 5,
        jurisdiction_filter: str = None,
        min_authority: int = 0,
    ) -> list:
        """Hybrid search combining semantic similarity with metadata filtering.

        Process:
        1. Vector similarity search with optional jurisdiction filter
        2. Filter by minimum authority level
        3. Re-rank by authority level (higher court = higher priority)
        4. Return top_k results

        Args:
            query: Natural language search query
            top_k: Maximum results to return
            jurisdiction_filter: Optional filter (e.g., 'federal', 'state')
            min_authority: Minimum authority level (0-10) to include

        Returns:
            List of dicts with case metadata, similarity score, and authority rank.

        Chapter Ref: Section 14.2.1, p.22-23
        Author: Imran Ahmad
        """
        # Build metadata filter
        meta_filter = {}
        if jurisdiction_filter:
            meta_filter["jurisdiction"] = jurisdiction_filter

        # Step 1: Semantic similarity search
        raw_results = self.vector_store.query(
            query_text=query,
            top_k=top_k * 2,  # Over-fetch to allow for filtering
            metadata_filter=meta_filter if meta_filter else None,
        )

        # Step 2: Filter by minimum authority
        filtered = [
            r for r in raw_results
            if r.metadata.get("authority_level", 0) >= min_authority
        ]

        # Step 3: Authority-weighted re-ranking
        # Combined score = 0.4 * normalized_similarity + 0.6 * normalized_authority
        for r in filtered:
            auth_normalized = r.metadata.get("authority_level", 0) / 10.0
            sim_normalized = max(0, (r.score + 1) / 2)  # Map [-1,1] to [0,1]
            r.score = 0.4 * sim_normalized + 0.6 * auth_normalized

        # Step 4: Sort by combined score and take top_k
        filtered.sort(key=lambda r: r.score, reverse=True)
        results = filtered[:top_k]

        # Format output
        formatted = []
        for rank, r in enumerate(results, 1):
            case_full = self.case_metadata.get(r.id, {})
            formatted.append({
                "rank": rank,
                "case_id": r.id,
                "case_name": r.metadata.get("case_name", "Unknown"),
                "citation": r.metadata.get("citation", "N/A"),
                "court": r.metadata.get("court", "Unknown"),
                "authority_level": r.metadata.get("authority_level", 0),
                "status": r.metadata.get("status", "unknown"),
                "relevance_score": round(r.score, 4),
                "key_issues": r.metadata.get("key_issues", []),
                "holding": case_full.get("holding", ""),
            })

        logger.success(f"Hybrid search: {len(formatted)} results for '{query[:50]}...'")
        return formatted

    def get_case(self, case_id: str) -> dict:
        """Retrieve full case details by ID."""
        return self.case_metadata.get(case_id, {})

    def count(self) -> int:
        """Return the number of cases in the knowledge base."""
        return self.vector_store.count()


# ── Initialize and ingest all cases ──
legal_kb = LegalKnowledgeBase()

print("\n" + "═" * 60)
print("  LEGAL KNOWLEDGE BASE — Section 14.2.1")
print("═" * 60)

for case in MOCK_LEGAL_CASES:
    legal_kb.ingest_case(case)

print(f"\n  Total cases ingested: {legal_kb.count()}")
print(f"  Verified cases: {sum(1 for c in MOCK_LEGAL_CASES if c['status'] == 'verified')}")
print(f"  Fabricated cases: {sum(1 for c in MOCK_LEGAL_CASES if c['status'] == 'fabricated')}")

# ── Demo: Hybrid search ──
print(f"\n{'─' * 60}")
print("  Hybrid Search: 'fiduciary duty investment advisor regulation'")
print(f"{'─' * 60}")

results = legal_kb.hybrid_search(
    query="fiduciary duty investment advisor regulation",
    top_k=5,
    jurisdiction_filter="federal",
    min_authority=1,  # Exclude fabricated case (authority=0)
)

for r in results:
    auth = r["authority_level"]
    auth_bar = "█" * auth + "░" * (10 - auth)
    print(f"\n  #{r['rank']}. {r['case_name']}")
    print(f"     Citation:  {r['citation']}")
    print(f"     Court:     {r['court'][:55]}")
    print(f"     Authority: [{auth_bar}] {auth}/10")
    print(f"     Score:     {r['relevance_score']:.4f}")
    print(f"     Status:    {r['status']}")

print(f"\n{'═' * 60}")
logger.success("Legal Knowledge Base operational")


[13:22:14] [Chapter14] INFO Building Legal Knowledge Base (Sec 14.2.1, p.20-23)

════════════════════════════════════════════════════════════
  LEGAL KNOWLEDGE BASE — Section 14.2.1
════════════════════════════════════════════════════════════
[13:22:14] [Chapter14] INFO MockVectorStore: Upserted document 'case_001'
[13:22:14] [Chapter14] INFO Ingested case: SEC v. Capital Growth Financial Services (authority: 8)
[13:22:14] [Chapter14] INFO MockVectorStore: Upserted document 'case_002'
[13:22:14] [Chapter14] INFO Ingested case: Transamerica Mortgage Advisors v. Lewis (authority: 10)
[13:22:14] [Chapter14] INFO MockVectorStore: Upserted document 'case_003'
[13:22:14] [Chapter14] INFO Ingested case: Goldstein v. SEC (authority: 8)
[13:22:14] [Chapter14] INFO MockVectorStore: Upserted document 'case_004'
[13:22:14] [Chapter14] INFO Ingested case: Jones v. Harris Associates L.P. (authority: 10)
[13:22:14] [Chapter14] INFO MockVectorStore: Upserted document 'case_005'
[13:22:14] [Chapter14] 

### Section 14.2.2 — Precedent Finding (Fig 14.2, p.23-27)

The `PrecedentFinder` implements a **3-stage pipeline** for legal precedent research:

```
  Stage 1: Issue Extraction     Stage 2: Precedent Search    Stage 3: Authority Ranking
  ┌────────────────────┐       ┌────────────────────┐       ┌────────────────────┐
  │  Parse legal query │  ──▸  │  For each issue,   │  ──▸  │  Rank by court     │
  │  into discrete     │       │  search knowledge  │       │  hierarchy and      │
  │  legal issues      │       │  base for relevant │       │  relevance score    │
  │  (Pydantic models) │       │  precedents        │       │                    │
  └────────────────────┘       └────────────────────┘       └────────────────────┘
```

`LegalIssue` and `IssueList` are Pydantic models ensuring structured, validated output at each stage. The LLM (or MockLLM in Simulation Mode) extracts issues from the query, then the knowledge base retrieves and ranks relevant cases.


In [11]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 10: Precedent Finding
# Chapter Ref: Section 14.2.2, Fig 14.2, p.23-27
# Book: 30 Agents Every AI Engineer Must Build — Imran Ahmad (Packt Publishing)
# ─────────────────────────────────────────────────────────────────────────────

logger.info("Building Precedent Finder (Sec 14.2.2, Fig 14.2, p.23-27)")

# ── Pydantic models for structured issue extraction ──

class LegalIssue(BaseModel):
    """A single legal issue extracted from a research query.

    Chapter Ref: Section 14.2.2, p.24
    Author: Imran Ahmad
    """
    issue: str = Field(description="A discrete legal issue or question")
    area_of_law: str = Field(description="The area of law this issue falls under")
    keywords: List[str] = Field(description="Key search terms for this issue")

class IssueList(BaseModel):
    """Structured list of legal issues from a query.

    Chapter Ref: Section 14.2.2, p.24
    Author: Imran Ahmad
    """
    issues: List[LegalIssue] = Field(description="Extracted legal issues")


class PrecedentFinder:
    """3-stage precedent finding pipeline.

    Stage 1: Extract discrete legal issues from the research query
    Stage 2: Search the knowledge base for precedents per issue
    Stage 3: Rank and deduplicate results by authority level

    Chapter Ref: Section 14.2.2, Fig 14.2, p.23-27
    Author: Imran Ahmad
    """

    def __init__(self, knowledge_base: LegalKnowledgeBase, llm_instance=None):
        self.kb = knowledge_base
        self.llm = llm_instance or llm

    @graceful_fallback(
        fallback_value=IssueList(issues=[
            LegalIssue(
                issue="Fiduciary duty of investment advisors",
                area_of_law="Securities Regulation",
                keywords=["fiduciary duty", "investment advisor", "SEC"]
            ),
            LegalIssue(
                issue="Excessive advisory fee standards",
                area_of_law="Investment Company Act",
                keywords=["advisory fees", "excessive fees", "mutual fund"]
            ),
        ]),
        section_ref="Sec 14.2.2"
    )
    def _extract_issues(self, query: str) -> IssueList:
        """Stage 1: Extract legal issues from the research query.

        Uses the LLM to parse a natural language legal query into structured
        LegalIssue objects. Falls back to predefined issues in Simulation Mode.

        Chapter Ref: Section 14.2.2, p.24-25
        Author: Imran Ahmad
        """
        extraction_prompt = f"""Analyze the following legal research query and extract
discrete legal issues. For each issue, identify the area of law and key search terms.

Query: {query}

Return a structured list of legal issues."""

        response = self.llm.invoke([HumanMessage(content=extraction_prompt)])

        # In simulation mode, return predefined issues based on keywords
        issues = []
        query_lower = query.lower()

        if any(kw in query_lower for kw in ["fiduciary", "advisor", "duty"]):
            issues.append(LegalIssue(
                issue="Fiduciary duty standards for investment advisors",
                area_of_law="Securities Regulation",
                keywords=["fiduciary duty", "investment advisor", "SEC enforcement"],
            ))
        if any(kw in query_lower for kw in ["fee", "excessive", "compensation"]):
            issues.append(LegalIssue(
                issue="Standards for evaluating excessive advisory fees",
                area_of_law="Investment Company Act",
                keywords=["advisory fees", "excessive fees", "Gartenberg standard"],
            ))
        if any(kw in query_lower for kw in ["fraud", "ponzi", "scheme"]):
            issues.append(LegalIssue(
                issue="Investor remedies in fraud and Ponzi scheme cases",
                area_of_law="Securities Fraud",
                keywords=["Ponzi scheme", "investor claims", "SIPA", "clawback"],
            ))
        if any(kw in query_lower for kw in ["hedge fund", "registration", "client"]):
            issues.append(LegalIssue(
                issue="Hedge fund advisor registration requirements",
                area_of_law="Investment Advisers Act",
                keywords=["hedge fund", "registration", "client definition"],
            ))

        # Default if no keywords match
        if not issues:
            issues.append(LegalIssue(
                issue="General investment regulation compliance",
                area_of_law="Securities Regulation",
                keywords=["investment regulation", "compliance", "SEC"],
            ))

        return IssueList(issues=issues)

    @graceful_fallback(
        fallback_value={"issues": [], "precedents": [], "total_found": 0},
        section_ref="Sec 14.2.2"
    )
    def find_precedents(self, query: str, top_k_per_issue: int = 3) -> dict:
        """Execute the full 3-stage precedent finding pipeline.

        Args:
            query: Natural language legal research query
            top_k_per_issue: Max precedents to retrieve per issue

        Returns:
            Dict with extracted issues, found precedents, and metadata.

        Chapter Ref: Section 14.2.2, p.25-27
        Author: Imran Ahmad
        """
        # Stage 1: Issue extraction
        logger.info("Stage 1: Extracting legal issues from query...")
        issue_list = self._extract_issues(query)
        logger.success(f"Extracted {len(issue_list.issues)} legal issues")

        # Stage 2: Search for precedents per issue
        logger.info("Stage 2: Searching knowledge base for precedents...")
        all_precedents = []
        seen_case_ids = set()

        for issue in issue_list.issues:
            search_query = f"{issue.issue} {' '.join(issue.keywords)}"
            results = self.kb.hybrid_search(
                query=search_query,
                top_k=top_k_per_issue,
                min_authority=1,  # Exclude fabricated cases
            )

            for r in results:
                if r["case_id"] not in seen_case_ids:
                    r["matched_issue"] = issue.issue
                    r["area_of_law"] = issue.area_of_law
                    all_precedents.append(r)
                    seen_case_ids.add(r["case_id"])

        # Stage 3: Final authority-weighted ranking
        logger.info("Stage 3: Ranking by court authority...")
        all_precedents.sort(key=lambda p: p["authority_level"], reverse=True)

        # Re-assign ranks after deduplication and sorting
        for i, p in enumerate(all_precedents, 1):
            p["final_rank"] = i

        logger.success(
            f"Precedent search complete: {len(all_precedents)} unique precedents "
            f"across {len(issue_list.issues)} issues"
        )

        return {
            "query": query,
            "issues": [
                {"issue": iss.issue, "area_of_law": iss.area_of_law, "keywords": iss.keywords}
                for iss in issue_list.issues
            ],
            "precedents": all_precedents,
            "total_found": len(all_precedents),
        }


# ── Demo: Precedent Finding ──
finder = PrecedentFinder(knowledge_base=legal_kb, llm_instance=llm)

print("\n" + "═" * 60)
print("  PRECEDENT FINDER — Section 14.2.2")
print("═" * 60)

research_query = (
    "What are the fiduciary duty standards for investment advisors, "
    "and what constitutes an excessive advisory fee?"
)
print(f"\n  Research Query:")
print(f"  '{research_query}'")

result = finder.find_precedents(research_query, top_k_per_issue=3)

print(f"\n  Extracted Issues ({len(result['issues'])}):")
for i, iss in enumerate(result["issues"], 1):
    print(f"    {i}. {iss['issue']}")
    print(f"       Area: {iss['area_of_law']} | Keywords: {', '.join(iss['keywords'][:3])}")

print(f"\n  Found Precedents ({result['total_found']}):")
for p in result["precedents"]:
    auth = p["authority_level"]
    auth_bar = "█" * auth + "░" * (10 - auth)
    print(f"\n    #{p['final_rank']}. {p['case_name']}")
    print(f"       Citation:  {p['citation']}")
    print(f"       Court:     {p['court'][:55]}")
    print(f"       Authority: [{auth_bar}] {auth}/10")
    print(f"       Issue:     {p['matched_issue'][:55]}")

print(f"\n{'═' * 60}")
logger.success("Precedent Finder operational")


[13:22:14] [Chapter14] INFO Building Precedent Finder (Sec 14.2.2, Fig 14.2, p.23-27)

════════════════════════════════════════════════════════════
  PRECEDENT FINDER — Section 14.2.2
════════════════════════════════════════════════════════════

  Research Query:
  'What are the fiduciary duty standards for investment advisors, and what constitutes an excessive advisory fee?'
[13:22:14] [Chapter14] INFO Stage 1: Extracting legal issues from query...
[13:22:14] [Chapter14] SUCCESS Extracted 2 legal issues
[13:22:14] [Chapter14] INFO Stage 2: Searching knowledge base for precedents...
[13:22:14] [Chapter14] SUCCESS Hybrid search: 3 results for 'Fiduciary duty standards for investment advisors f...'
[13:22:14] [Chapter14] SUCCESS Hybrid search: 3 results for 'Standards for evaluating excessive advisory fees a...'
[13:22:14] [Chapter14] INFO Stage 3: Ranking by court authority...
[13:22:14] [Chapter14] SUCCESS Precedent search complete: 3 unique precedents across 2 issues

  Extracted Issu

### Section 14.2.3 — Contract Analysis Agent (Fig 14.3, p.27-30)

The `ContractAnalysisAgent` implements a **5-stage pipeline** for comprehensive contract review:

```
  Stage 1          Stage 2          Stage 3           Stage 4          Stage 5
  ┌──────────┐    ┌──────────┐    ┌──────────┐     ┌──────────┐    ┌──────────┐
  │ Clause   │ ▸  │ Risk     │ ▸  │ Compliance│ ▸   │ Missing  │ ▸  │ Summary  │
  │ Classif. │    │ Flagging │    │ Check    │     │ Provision│    │ Report   │
  └──────────┘    └──────────┘    └──────────┘     └──────────┘    └──────────┘
```

The analysis of `MOCK_CONTRACT` should identify:
- **HIGH risk** in Sections 4 (indemnification) and 5 (liability cap)
- **CRITICAL** missing GDPR Data Processing Addendum
- **MEDIUM** risk in Section 7 (unilateral termination)


In [12]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 11: Contract Analysis Agent
# Chapter Ref: Section 14.2.3, Fig 14.3, p.27-30
# Book: 30 Agents Every AI Engineer Must Build — Imran Ahmad (Packt Publishing)
# ─────────────────────────────────────────────────────────────────────────────

logger.info("Building Contract Analysis Agent (Sec 14.2.3, Fig 14.3, p.27-30)")

class ContractAnalysisAgent:
    """5-stage contract analysis pipeline.

    Stage 1: Clause Classification — categorize each clause by type
    Stage 2: Risk Flagging — identify high-risk provisions
    Stage 3: Compliance Check — verify regulatory requirements
    Stage 4: Missing Provisions — detect absent but required clauses
    Stage 5: Summary Report — consolidate findings into actionable output

    Chapter Ref: Section 14.2.3, Fig 14.3, p.27-30
    Author: Imran Ahmad
    """

    RISK_PRIORITY = {"CRITICAL": 0, "HIGH": 1, "MEDIUM": 2, "LOW": 3}

    def __init__(self, llm_instance=None):
        self.llm = llm_instance or llm

    @graceful_fallback(
        fallback_value={"stages_completed": 0, "error": "Analysis failed"},
        section_ref="Sec 14.2.3"
    )
    def analyze(self, contract: dict) -> dict:
        """Run the full 5-stage contract analysis pipeline.

        Args:
            contract: Contract dict with 'clauses' and 'missing_provisions' keys.

        Returns:
            Comprehensive analysis report with findings from all 5 stages.

        Chapter Ref: Section 14.2.3, p.27-30
        Author: Imran Ahmad
        """
        logger.info(f"Analyzing contract: {contract.get('title', 'Unknown')}")

        # Stage 1: Clause Classification
        logger.info("Stage 1/5: Clause Classification...")
        clause_analysis = self._classify_clauses(contract["clauses"])

        # Stage 2: Risk Flagging
        logger.info("Stage 2/5: Risk Flagging...")
        risk_findings = self._flag_risks(contract["clauses"])

        # Stage 3: Compliance Check
        logger.info("Stage 3/5: Compliance Check...")
        compliance_findings = self._check_compliance(contract["clauses"])

        # Stage 4: Missing Provisions
        logger.info("Stage 4/5: Missing Provision Detection...")
        missing = contract.get("missing_provisions", [])
        missing_findings = self._analyze_missing_provisions(missing)

        # Stage 5: Summary Report
        logger.info("Stage 5/5: Generating Summary Report...")
        report = self._generate_report(
            contract, clause_analysis, risk_findings,
            compliance_findings, missing_findings
        )

        logger.success("Contract analysis complete (5/5 stages)")
        return report

    def _classify_clauses(self, clauses: list) -> list:
        """Stage 1: Classify clauses by type and risk level."""
        analysis = []
        for clause in clauses:
            analysis.append({
                "section": clause["section"],
                "title": clause["title"],
                "classification": clause.get("classification", "unknown"),
                "risk_level": clause.get("risk_level", "LOW"),
            })
        return analysis

    def _flag_risks(self, clauses: list) -> list:
        """Stage 2: Identify and detail high-risk provisions."""
        risks = []
        for clause in clauses:
            if clause.get("risk_level") in ("HIGH", "CRITICAL"):
                risks.append({
                    "section": clause["section"],
                    "title": clause["title"],
                    "risk_level": clause["risk_level"],
                    "risk_note": clause.get("risk_note", "Elevated risk identified"),
                })
            elif clause.get("risk_level") == "MEDIUM" and clause.get("risk_note"):
                risks.append({
                    "section": clause["section"],
                    "title": clause["title"],
                    "risk_level": clause["risk_level"],
                    "risk_note": clause.get("risk_note", ""),
                })
        # Sort by severity
        risks.sort(key=lambda r: self.RISK_PRIORITY.get(r["risk_level"], 99))
        return risks

    def _check_compliance(self, clauses: list) -> list:
        """Stage 3: Check clauses for compliance issues."""
        findings = []
        for clause in clauses:
            if clause.get("compliance_note"):
                findings.append({
                    "section": clause["section"],
                    "title": clause["title"],
                    "compliance_note": clause["compliance_note"],
                })
        return findings

    def _analyze_missing_provisions(self, missing: list) -> list:
        """Stage 4: Analyze missing provisions and their impact."""
        findings = []
        for provision in missing:
            findings.append({
                "provision": provision["provision"],
                "severity": provision["severity"],
                "description": provision["description"],
            })
        findings.sort(key=lambda f: self.RISK_PRIORITY.get(f["severity"], 99))
        return findings

    def _generate_report(self, contract, clauses, risks, compliance, missing) -> dict:
        """Stage 5: Consolidate all findings into a summary report."""
        # Count by risk level
        risk_counts = {}
        for c in contract["clauses"]:
            rl = c.get("risk_level", "LOW")
            risk_counts[rl] = risk_counts.get(rl, 0) + 1

        return {
            "contract_id": contract.get("contract_id", "Unknown"),
            "title": contract.get("title", "Unknown"),
            "parties": contract.get("parties", {}),
            "stages_completed": 5,
            "clause_analysis": clauses,
            "risk_findings": risks,
            "compliance_findings": compliance,
            "missing_provisions": missing,
            "summary": {
                "total_clauses": len(contract["clauses"]),
                "risk_distribution": risk_counts,
                "high_risk_count": len(risks),
                "compliance_issues": len(compliance),
                "missing_provisions_count": len(missing),
                "overall_risk_rating": "HIGH" if any(
                    r["risk_level"] in ("HIGH", "CRITICAL") for r in risks
                ) else "MODERATE" if risks else "LOW",
            },
        }


# ── Demo: Analyze MOCK_CONTRACT ──
contract_agent = ContractAnalysisAgent(llm_instance=llm)
report = contract_agent.analyze(MOCK_CONTRACT)

print("\n" + "═" * 60)
print("  CONTRACT ANALYSIS — Section 14.2.3")
print("═" * 60)
print(f"  Contract: {report['title']} ({report['contract_id']})")
print(f"  Parties:  {report['parties'].get('party_a', 'N/A')}")
print(f"         vs {report['parties'].get('party_b', 'N/A')}")

# Summary
s = report["summary"]
print(f"\n  ┌─ ANALYSIS SUMMARY ────────────────────────────────────┐")
print(f"  │ Total Clauses:       {s['total_clauses']:>3}                               │")
print(f"  │ Risk Distribution:   {s['risk_distribution']}  │")
print(f"  │ High-Risk Findings:  {s['high_risk_count']:>3}                               │")
print(f"  │ Compliance Issues:   {s['compliance_issues']:>3}                               │")
print(f"  │ Missing Provisions:  {s['missing_provisions_count']:>3}                               │")
print(f"  │ Overall Risk Rating: {s['overall_risk_rating']:>8}                          │")
print(f"  └────────────────────────────────────────────────────────┘")

# Risk findings
print(f"\n  Risk Findings:")
for r in report["risk_findings"]:
    icon = "🔴" if r["risk_level"] == "HIGH" else "🟡" if r["risk_level"] == "MEDIUM" else "🟢"
    print(f"    {icon} [{r['risk_level']:>6}] Section {r['section']}: {r['title']}")
    if r.get("risk_note"):
        # Print first line of risk note
        note_line = r["risk_note"].split(".")[0] + "."
        print(f"          {note_line[:65]}")

# Missing provisions
print(f"\n  Missing Provisions:")
for m in report["missing_provisions"]:
    icon = "🔴" if m["severity"] == "CRITICAL" else "🟡" if m["severity"] == "MEDIUM" else "🟢"
    print(f"    {icon} [{m['severity']:>8}] {m['provision']}")

# Compliance
print(f"\n  Compliance Issues:")
for c in report["compliance_findings"]:
    print(f"    ⚠  Section {c['section']}: {c['title']}")
    note_line = c["compliance_note"].split(".")[0] + "."
    print(f"       {note_line[:65]}")

print(f"\n{'═' * 60}")
logger.success("Contract Analysis Agent operational")


[13:22:15] [Chapter14] INFO Building Contract Analysis Agent (Sec 14.2.3, Fig 14.3, p.27-30)
[13:22:15] [Chapter14] INFO Analyzing contract: Master Services Agreement
[13:22:15] [Chapter14] INFO Stage 1/5: Clause Classification...
[13:22:15] [Chapter14] INFO Stage 2/5: Risk Flagging...
[13:22:15] [Chapter14] INFO Stage 3/5: Compliance Check...
[13:22:15] [Chapter14] INFO Stage 4/5: Missing Provision Detection...
[13:22:15] [Chapter14] INFO Stage 5/5: Generating Summary Report...
[13:22:15] [Chapter14] SUCCESS Contract analysis complete (5/5 stages)

════════════════════════════════════════════════════════════
  CONTRACT ANALYSIS — Section 14.2.3
════════════════════════════════════════════════════════════
  Contract: Master Services Agreement (SVC-2024-00847)
  Parties:  Acme Financial Technologies, Inc.
         vs GlobalData Analytics, Ltd.

  ┌─ ANALYSIS SUMMARY ────────────────────────────────────┐
  │ Total Clauses:         8                               │
  │ Risk Distribution: 

### Section 14.2.4 — LegalBrief Case Study: Citation Verification (p.31-33)

This cell demonstrates the complete legal research workflow with a critical safety feature: **citation verification**. The `verify_citations()` function checks every cited case against the knowledge base to detect hallucinated references.

The workflow is implemented as a 5-node `StateGraph`:

```
  ┌──────────┐    ┌──────────┐    ┌──────────┐    ┌──────────┐    ┌──────────┐
  │ Research  │ ▸  │ Find     │ ▸  │ Draft    │ ▸  │ Verify   │ ▸  │ Finalize │
  │ Query    │    │ Precedent│    │ Brief    │    │ Citations│    │ Brief    │
  └──────────┘    └──────────┘    └──────────┘    └──────────┘    └──────────┘
```

The fabricated `Varghese v. Tech Corp International` case (status: "fabricated", authority: 0) is specifically designed to be caught by the verification stage, demonstrating why this pipeline step is essential.


In [13]:
# ─────────────────────────────────────────────────────────────────────────────
# Cell 12: LegalBrief Case Study — Citation Verification
# Chapter Ref: Section 14.2.4, p.31-33
# Book: 30 Agents Every AI Engineer Must Build — Imran Ahmad (Packt Publishing)
# ─────────────────────────────────────────────────────────────────────────────

logger.info("Building LegalBrief Case Study (Sec 14.2.4, p.31-33)")

# ── Legal Research State ──
class LegalResearchState(TypedDict):
    """Shared state for the legal research workflow.

    Chapter Ref: Section 14.2.4, p.31
    Author: Imran Ahmad
    """
    research_query: str
    extracted_issues: list
    precedents: list
    draft_brief: str
    cited_cases: list
    verified_citations: list
    unverified_citations: list
    final_brief: str
    messages: list

# ── Node 1: Research Query ──
def research_query_node(state: dict) -> dict:
    """Parse and validate the incoming research query.

    Chapter Ref: Section 14.2.4, p.31
    Author: Imran Ahmad
    """
    query = state["research_query"]
    logger.info(f"Node 1/5: Research query received: '{query[:60]}...'")
    return {
        "messages": [AIMessage(content=f"Research query parsed: {query[:80]}")],
    }

# ── Node 2: Find Precedents ──
@graceful_fallback(
    fallback_value={"precedents": [], "extracted_issues": [], "messages": []},
    section_ref="Sec 14.2.4"
)
def find_precedent_node(state: dict) -> dict:
    """Find relevant precedents using the PrecedentFinder.

    Chapter Ref: Section 14.2.4, p.31-32
    Author: Imran Ahmad
    """
    logger.info("Node 2/5: Finding precedents...")
    result = finder.find_precedents(state["research_query"], top_k_per_issue=3)
    return {
        "precedents": result["precedents"],
        "extracted_issues": result["issues"],
        "messages": [AIMessage(
            content=f"Found {result['total_found']} precedents across {len(result['issues'])} issues"
        )],
    }

# ── Node 3: Draft Brief ──
@graceful_fallback(
    fallback_value={"draft_brief": "Draft unavailable", "cited_cases": [], "messages": []},
    section_ref="Sec 14.2.4"
)
def draft_brief_node(state: dict) -> dict:
    """Draft a legal brief citing the found precedents.

    Also intentionally includes the fabricated Varghese case to
    demonstrate the citation verification pipeline.

    Chapter Ref: Section 14.2.4, p.32
    Author: Imran Ahmad
    """
    logger.info("Node 3/5: Drafting legal brief...")
    precedents = state.get("precedents", [])

    # Build citations from found precedents
    cited_cases = []
    brief_sections = []

    for p in precedents:
        cited_cases.append({
            "case_id": p["case_id"],
            "case_name": p["case_name"],
            "citation": p["citation"],
            "authority_level": p["authority_level"],
        })
        brief_sections.append(
            f"As established in {p['case_name']} ({p['citation']}), "
            f"the court held that {p.get('holding', 'relevant legal principle applies')}."
        )

    # Intentionally include the fabricated case to test verification
    fabricated = next(
        (c for c in MOCK_LEGAL_CASES if c["status"] == "fabricated"), None
    )
    if fabricated:
        cited_cases.append({
            "case_id": fabricated["case_id"],
            "case_name": fabricated["case_name"],
            "citation": fabricated["citation"],
            "authority_level": fabricated["authority_level"],
        })
        brief_sections.append(
            f"Furthermore, in {fabricated['case_name']} ({fabricated['citation']}), "
            f"the court addressed related issues of trade secret protection."
        )
        logger.warning("Fabricated case 'Varghese v. Tech Corp' included for verification testing")

    draft = "\n\n".join(brief_sections) if brief_sections else "No precedents available for brief."

    return {
        "draft_brief": draft,
        "cited_cases": cited_cases,
        "messages": [AIMessage(content=f"Brief drafted with {len(cited_cases)} citations")],
    }

# ── Node 4: Verify Citations ──
@graceful_fallback(
    fallback_value={"verified_citations": [], "unverified_citations": [], "messages": []},
    section_ref="Sec 14.2.4"
)
def verify_citations_node(state: dict) -> dict:
    """Verify each citation against the knowledge base.

    Checks:
    1. Case exists in the knowledge base
    2. Case status is 'verified' (not 'fabricated')
    3. Citation string matches the record

    This is the critical safety feature inspired by the Schwartz v. Avianca
    incident. Any citation that fails verification is flagged and excluded
    from the final brief.

    Chapter Ref: Section 14.2.4, p.32-33
    Author: Imran Ahmad
    """
    logger.info("Node 4/5: Verifying citations...")
    cited = state.get("cited_cases", [])
    verified = []
    unverified = []

    for cite in cited:
        case_id = cite["case_id"]
        case_record = legal_kb.get_case(case_id)

        if not case_record:
            reason = "Case not found in knowledge base"
            unverified.append({**cite, "reason": reason})
            logger.error(f"UNVERIFIED: {cite['case_name']} — {reason}")
        elif case_record.get("status") == "fabricated":
            reason = "Case flagged as FABRICATED — does not exist in court records"
            unverified.append({**cite, "reason": reason})
            logger.error(f"UNVERIFIED: {cite['case_name']} — {reason}")
        elif case_record.get("citation") != cite.get("citation"):
            reason = "Citation string does not match record"
            unverified.append({**cite, "reason": reason})
            logger.error(f"UNVERIFIED: {cite['case_name']} — {reason}")
        else:
            verified.append(cite)
            logger.success(f"VERIFIED: {cite['case_name']}")

    return {
        "verified_citations": verified,
        "unverified_citations": unverified,
        "messages": [AIMessage(
            content=f"Citation verification: {len(verified)} verified, {len(unverified)} rejected"
        )],
    }

# ── Node 5: Finalize Brief ──
@graceful_fallback(
    fallback_value={"final_brief": "Brief finalization failed", "messages": []},
    section_ref="Sec 14.2.4"
)
def finalize_brief_node(state: dict) -> dict:
    """Finalize the brief using only verified citations.

    Removes any references to unverified/fabricated cases and produces
    the final research brief.

    Chapter Ref: Section 14.2.4, p.33
    Author: Imran Ahmad
    """
    logger.info("Node 5/5: Finalizing brief with verified citations only...")
    verified = state.get("verified_citations", [])
    unverified = state.get("unverified_citations", [])

    sections = []
    sections.append("LEGAL RESEARCH BRIEF")
    sections.append("=" * 40)
    sections.append(f"Research Query: {state.get('research_query', 'N/A')}")
    sections.append("")

    if verified:
        sections.append("VERIFIED CITATIONS:")
        for v in verified:
            sections.append(f"  ✓ {v['case_name']}")
            sections.append(f"    {v['citation']}")
            sections.append(f"    Authority Level: {v['authority_level']}/10")
            sections.append("")

    if unverified:
        sections.append("REJECTED CITATIONS (excluded from brief):")
        for u in unverified:
            sections.append(f"  ✗ {u['case_name']}")
            sections.append(f"    Reason: {u['reason']}")
            sections.append("")

    sections.append(f"Total citations: {len(verified)} verified, {len(unverified)} rejected")
    final = "\n".join(sections)

    return {
        "final_brief": final,
        "messages": [AIMessage(content="Legal brief finalized")],
    }


# ── Build the LegalBrief StateGraph ──
legal_workflow = StateGraph(LegalResearchState)

legal_workflow.add_node("parse_query", research_query_node)
legal_workflow.add_node("search_cases", find_precedent_node)
legal_workflow.add_node("compose_brief", draft_brief_node)
legal_workflow.add_node("check_citations", verify_citations_node)
legal_workflow.add_node("produce_brief", finalize_brief_node)

legal_workflow.set_entry_point("parse_query")
legal_workflow.add_edge("parse_query", "search_cases")
legal_workflow.add_edge("search_cases", "compose_brief")
legal_workflow.add_edge("compose_brief", "check_citations")
legal_workflow.add_edge("check_citations", "produce_brief")
legal_workflow.add_edge("produce_brief", END)

legal_graph = legal_workflow.compile()
logger.success("LegalBrief StateGraph compiled (5 nodes)")

# ── Execute the LegalBrief Case Study ──
print("\n" + "█" * 60)
print("  LEGAL BRIEF CASE STUDY — Section 14.2.4")
print("  Citation Verification Pipeline Demo")
print("█" * 60)

research_state = {
    "research_query": (
        "What are the fiduciary duty standards for investment advisors "
        "and what constitutes excessive advisory fees?"
    ),
    "extracted_issues": [],
    "precedents": [],
    "draft_brief": "",
    "cited_cases": [],
    "verified_citations": [],
    "unverified_citations": [],
    "final_brief": "",
    "messages": [],
}

# Stream execution
final_output = {}
for step in legal_graph.stream(research_state):
    for node_name, output in step.items():
        print(f"\n  ▸ Node: {node_name}")
        msgs = output.get("messages", [])
        for msg in msgs:
            if hasattr(msg, "content"):
                print(f"    {msg.content}")
        final_output.update(output)

# Print the final brief
print(f"\n{'─' * 60}")
print(final_output.get("final_brief", "No brief generated"))
print(f"{'─' * 60}")

# Highlight the Varghese detection
unverified = final_output.get("unverified_citations", [])
if unverified:
    print(f"\n  ⚠  CITATION VERIFICATION ALERT:")
    for u in unverified:
        print(f"     Case: {u['case_name']}")
        print(f"     Reason: {u['reason']}")
        print(f"     This demonstrates the Schwartz v. Avianca safeguard (Sec 14.2.4)")

print(f"\n{'█' * 60}")
logger.success("LegalBrief Case Study complete — fabricated citation detected and rejected")
logger.success("Legal Intelligence Agent (Section 14.2) COMPLETE")


[13:22:15] [Chapter14] INFO Building LegalBrief Case Study (Sec 14.2.4, p.31-33)
[13:22:15] [Chapter14] SUCCESS LegalBrief StateGraph compiled (5 nodes)

████████████████████████████████████████████████████████████
  LEGAL BRIEF CASE STUDY — Section 14.2.4
  Citation Verification Pipeline Demo
████████████████████████████████████████████████████████████
[13:22:15] [Chapter14] INFO Node 1/5: Research query received: 'What are the fiduciary duty standards for investment advisor...'



  ▸ Node: parse_query


    Research query parsed: What are the fiduciary duty standards for investment advisors and what constitut
[13:22:15] [Chapter14] INFO Node 2/5: Finding precedents...
[13:22:15] [Chapter14] INFO Stage 1: Extracting legal issues from query...
[13:22:15] [Chapter14] SUCCESS Extracted 2 legal issues
[13:22:15] [Chapter14] INFO Stage 2: Searching knowledge base for precedents...
[13:22:15] [Chapter14] SUCCESS Hybrid search: 3 results for 'Fiduciary duty standards for investment advisors f...'
[13:22:15] [Chapter14] SUCCESS Hybrid search: 3 results for 'Standards for evaluating excessive advisory fees a...'
[13:22:15] [Chapter14] INFO Stage 3: Ranking by court authority...
[13:22:15] [Chapter14] SUCCESS Precedent search complete: 3 unique precedents across 2 issues

  ▸ Node: search_cases
    Found 3 precedents across 2 issues
[13:22:15] [Chapter14] INFO Node 3/5: Drafting legal brief...
[13:22:15] [Chapter14] WARNING Fabricated case 'Varghese v. Tech Corp' included for verification testi

---

## Summary & Extension Ideas (p.33-34)

### Key Takeaways from Chapter 14

**Financial Advisory Agent (Section 14.1):**
- The **supervisor pattern** (Fig 14.1) provides centralized orchestration, preventing independent agent actions that could bypass compliance requirements
- **Quantitative risk metrics** (VaR, CVaR, volatility, max drawdown) provide objective portfolio assessment beyond simple heuristics
- **Compliance-by-architecture** enforces regulatory requirements as structural workflow gates, not optional review steps — directly addressing the Knight Capital lesson
- The **inter-agent message protocol** (p.19) enables transparent communication between planning and compliance agents

**Legal Intelligence Agent (Section 14.2):**
- **Hybrid retrieval** (keyword + semantic) achieves higher recall than either method alone for legal research
- **Authority-weighted ranking** ensures Supreme Court rulings outrank district court opinions regardless of semantic similarity
- **Citation verification** is a critical safety layer that detects hallucinated case references — a lesson from the Schwartz v. Avianca incident
- The **5-stage contract analysis pipeline** (Fig 14.3) provides systematic, reproducible contract review

### Extension Ideas

1. **Add more stock symbols** — Extend `MOCK_STOCK_DATA` with additional symbols and test the risk scoring across a larger portfolio
2. **Connect a real vector database** — Replace `MockVectorStore` with Pinecone, Weaviate, or ChromaDB for production-scale legal research
3. **Add more legal cases** — Extend `MOCK_LEGAL_CASES` with additional case law from different jurisdictions
4. **Implement real-time data feeds** — Connect to live market data websockets for streaming price updates
5. **Multi-jurisdiction support** — Add state-level legal cases and jurisdiction-specific compliance rules
6. **Enhanced contract templates** — Add more contract types (NDA, employment, licensing) to the analysis pipeline
7. **Audit trail logging** — Add persistent logging of all agent decisions for regulatory compliance reporting

---

### About This Repository

**Book:** *30 Agents Every AI Engineer Must Build*
**Author:** Imran Ahmad
**Publisher:** Packt Publishing
**Chapter:** 14 — Financial and Legal Domain Agents

This notebook demonstrates that production-grade AI agents in regulated domains require not just intelligence (LLM reasoning), but also structural safety (compliance gates, citation verification, risk scoring) and operational resilience (simulation mode, graceful fallback, defensive logging).

---

*End of Chapter 14 Notebook — Imran Ahmad*
